# Online Retail — Câu 7: Chuyển hot encoding thành one-hot encoding

**Võ Thiện Linh — MSSV: 6451071041**

Từ ma trận số lượng, đổi thành ma trận nhị phân: `1` nếu tổng số lượng mặt hàng trong hóa đơn lớn hơn 0, ngược lại là `0`. Ma trận lưu dạng thưa để giảm bộ nhớ.

In [ ]:
print('Võ Thiện Linh - MSSV: 6451071041')
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
candidates = [Path('Online Retail.xlsx'), Path('LAB1') / 'Phan02' / '6451071041_LAB1_Bai2_Trang37' / 'Online Retail.xlsx', Path('/content/Online Retail.xlsx')]
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    raise FileNotFoundError(f'Không tìm thấy Online Retail.xlsx. Thư mục đang chạy: {Path.cwd()}')
df = pd.read_excel(data_path, engine='openpyxl')
df['Description'] = df['Description'].str.strip()
df = df.dropna(subset=['InvoiceNo']).copy()
df['InvoiceNo'] = df['InvoiceNo'].astype('string').str.strip()
df = df.loc[~df['InvoiceNo'].str.contains('C', regex=False, na=False)]
uk = df.loc[df['Country'].eq('United Kingdom')]
grouped_quantity = uk.groupby(['InvoiceNo', 'Description'], as_index=False, dropna=True)['Quantity'].sum()
invoice_index = pd.Index(grouped_quantity['InvoiceNo'].drop_duplicates(), name='InvoiceNo')
row_position = {invoice: i for i, invoice in enumerate(invoice_index)}
sparse_columns = {}
for description, rows in grouped_quantity.groupby('Description', sort=False):
    values = np.zeros(len(invoice_index), dtype=bool)
    positions = rows['InvoiceNo'].map(row_position).to_numpy(dtype=np.intp)
    values[positions] = rows['Quantity'].to_numpy() > 0
    sparse_columns[description] = pd.arrays.SparseArray(values, fill_value=False)
basket_onehot = pd.DataFrame(sparse_columns, index=invoice_index)
print(f'One-hot matrix: {basket_onehot.shape[0]:,} hóa đơn × {basket_onehot.shape[1]:,} mặt hàng')
print('Kiểu dữ liệu cột:', basket_onehot.dtypes.iloc[0])
print(f'Bộ nhớ ma trận thưa: {basket_onehot.memory_usage(index=True, deep=True).sum() / 1024**2:.2f} MiB')
print('Một số mặt hàng và hóa đơn:')
sample_columns = list(basket_onehot.columns[:10])
display(basket_onehot.loc[:, sample_columns].iloc[:8].sparse.to_dense().astype(int))
print('Giá trị: 1 = có mua (số lượng tổng > 0), 0 = không mua/không có số lượng dương.')